<a href="https://colab.research.google.com/github/alimoorreza/CS181-fall26-Notebook-5/blob/main/cs181_part1_training_HRNet_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#**Part#1: Train a semantic segmentation model (HRNet)**
In this task, you will train a state-of-the-art semantic segmentation model known as **[HRNet](https://arxiv.org/pdf/1908.07919) (*High Resolution Net*)**. They have demonstrated superior performance compared to the widely recognized and commonly utilized models discussed in class, including:
* **FCN**
* **SegNet**
* **PSPNet**
* **U-Net**

In [1]:
from google.colab import drive
drive.mount('/content/drive/')

Mounted at /content/drive/


##**Install the necessary libraries**

In [2]:
import torch
print(torch.__version__)

2.14.1+cu130


In [3]:
import torchvision
print(torchvision.__version__)

0.29.1+cu130


In [4]:
! pip install gdown

In [5]:
%%shell
pip install imagecodecs
pip install pickle5
pip install timm
pip install tensorboardX
pip install yacs

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 28.4/28.4 MB 49.2 MB/s  0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  error: subprocess-exited-with-error
  
  × Building wheel for pickle5 (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> No available output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  ERROR: Failed building wheel for pickle5
Failed to build pickle5
error: failed-wheel-build-for-install

× Failed to build installable wheels for some pyproject.toml based projects
╰─> pickle5


In [6]:
%%shell
[[ -d ./hrnetv2_sem_seg ]]
rm -rf ./hrnetv2_sem_seg
git clone https://github.com/alimoorreza/hrnetv2_sem_seg

Cloning into 'hrnetv2_sem_seg'...
remote: Enumerating objects: 2363, done.
remote: Counting objects: 100% (144/144), done.
remote: Compressing objects: 100% (58/58), done.
remote: Total 2363 (delta 116), reused 83 (delta 83), pack-reused 2219 (from 1)
Receiving objects: 100% (2363/2363), 181.29 MiB | 28.48 MiB/s, done.
Resolving deltas: 100% (225/225), done.
Updating files: 100% (1983/1983), done.


In [7]:
%%shell
mkdir -p hrnetv2_sem_seg/log
mkdir -p hrnetv2_sem_seg/output
mkdir -p hrnetv2_sem_seg/init_model
mkdir -p hrnetv2_sem_seg/dataset

##**Download the [UWSv1 dataset](https://drive.google.com/uc?id=155Vg3iOTK3aed11LXC2lFQBxhdimfuZB)**
* This dataset comprises 574 images featuring 21 underwater animal categories and background objects such as *water and rock*. Pixel-wise annotations were provided by in-house annotators from Indiana University and Drake University, encompassing 29 total categories. The dataset is divided into training (461 images) and testing (113 images) sets using an 80:20 random split.

In [8]:
%%shell
gdown https://drive.google.com/uc?id=155Vg3iOTK3aed11LXC2lFQBxhdimfuZB

Downloading...
From (original): https://drive.google.com/uc?id=155Vg3iOTK3aed11LXC2lFQBxhdimfuZB
From (redirected): https://drive.google.com/uc?id=155Vg3iOTK3aed11LXC2lFQBxhdimfuZB&confirm=t&uuid=d6ff69f8-e2d5-4c85-abeb-b7c25dfb5aee
To: /content/uw_sem_seg_dataset.zip
100% 571M/571M [00:05<00:00, 95.7MB/s]


In [9]:
%%shell
unzip /content/uw_sem_seg_dataset.zip  -d hrnetv2_sem_seg/dataset

Archive:  /content/uw_sem_seg_dataset.zip
   creating: hrnetv2_sem_seg/dataset/uw_sem_seg_dataset/
  inflating: hrnetv2_sem_seg/dataset/__MACOSX/._uw_sem_seg_dataset  
   creating: hrnetv2_sem_seg/dataset/uw_sem_seg_dataset/train/
  inflating: hrnetv2_sem_seg/dataset/__MACOSX/uw_sem_seg_dataset/._train  
  inflating: hrnetv2_sem_seg/dataset/uw_sem_seg_dataset/._train  
   creating: hrnetv2_sem_seg/dataset/uw_sem_seg_dataset/validation/
  inflating: hrnetv2_sem_seg/dataset/__MACOSX/uw_sem_seg_dataset/._validation  
  inflating: hrnetv2_sem_seg/dataset/uw_sem_seg_dataset/._validation  
   creating: hrnetv2_sem_seg/dataset/uw_sem_seg_dataset/train/images/
  inflating: hrnetv2_sem_seg/dataset/__MACOSX/uw_sem_seg_dataset/train/._images  
  inflating: hrnetv2_sem_seg/dataset/uw_sem_seg_dataset/train/._images  
   creating: hrnetv2_sem_seg/dataset/uw_sem_seg_dataset/train/labels/
  inflating: hrnetv2_sem_seg/dataset/__MACOSX/uw_sem_seg_dataset/train/._labels  
  inflating: hrnetv2_sem_seg/dat

##**Download the pretrained weights for hrnet-v2**

In [10]:
%%shell
gdown https://drive.google.com/uc?id=1CHOKZcn6TnThoajYmfwjJ9g7Q4SRW4k4
mv hrnetv2_w48_imagenet_pretrained.pth hrnetv2_sem_seg/init_model/


Downloading...
From (original): https://drive.google.com/uc?id=1CHOKZcn6TnThoajYmfwjJ9g7Q4SRW4k4
From (redirected): https://drive.google.com/uc?id=1CHOKZcn6TnThoajYmfwjJ9g7Q4SRW4k4&confirm=t&uuid=8983fbc5-c88b-4fe0-a33e-3fb463d82f16
To: /content/hrnetv2_w48_imagenet_pretrained.pth
100% 311M/311M [00:04<00:00, 73.6MB/s]


##**Cleanup folder names such as .DS_store**

In [ ]:
%%shell
find ./hrnetv2_sem_seg -name "._*" -delete
find ./hrnetv2_sem_seg/dataset/ -name "__MACOSX" -exec rm -rf {} +

##**Run the training script**

In [11]:
%%shell
python3 ./hrnetv2_sem_seg/tools/train_hrnet_v2_uws.py \
    --cfg ./hrnetv2_sem_seg/experiments/hrnet/uws_training_hrnet_v2_train_COLAB.yaml


Seeding with 304
=> creating /content/hrnetv2_sem_seg/output/UWS3/uws_training_hrnet_v2_train_COLAB
=> creating /content/hrnetv2_sem_seg/log/UWS3/seg_hrnet/uws_training_hrnet_v2_train_COLAB_2026-11-11-19-49
Namespace(cfg='./hrnetv2_sem_seg/experiments/hrnet/uws_training_hrnet_v2_train_COLAB.yaml', seed=304, opts=[])
AUTO_RESUME: False
CUDNN:
  BENCHMARK: True
  DETERMINISTIC: False
  ENABLED: True
DATASET:
  DATASET: UWS3
  EXTRA_TRAIN_SET: 
  NUM_CLASSES: 29
  ROOT: /content/hrnetv2_sem_seg/dataset/uw_sem_seg_dataset/
  TEST_SET: validation
  TRAIN_SET: train
DEBUG:
  DEBUG: False
  SAVE_BATCH_IMAGES_GT: False
  SAVE_BATCH_IMAGES_PRED: False
  SAVE_HEATMAPS_GT: False
  SAVE_HEATMAPS_PRED: False
GPUS: (0,)
LOG_DIR: /content/hrnetv2_sem_seg//log
LOSS:
  BALANCE_WEIGHTS: [1]
  CLASS_BALANCE: False
  OHEMKEEP: 131072
  OHEMTHRES: 0.9
  USE_OHEM: True
MODEL:
  ALIGN_CORNERS: True
  EXTRA:
    FINAL_CONV_KERNEL: 1
    STAGE1:
      BLOCK: BOTTLENECK
      FUSE_METHOD: SUM
      NUM_BLOCKS: 

CalledProcessError: Command 'python3 ./hrnetv2_sem_seg/tools/train_hrnet_v2_uws.py \
    --cfg ./hrnetv2_sem_seg/experiments/hrnet/uws_training_hrnet_v2_train_COLAB.yaml
' returned non-zero exit status 1.

# __Task 1.1:__ (3 points)
### **Training HRNet semantic segmentation model using UWSv1 dataset.**
- **Step 1:** Finish training [HRNet v2 model](https://arxiv.org/pdf/1908.07919)
- **Step 2:** The performance of the semantic segmentation task is evaluated using the **mean Intersection over Union (mIoU)** metric. Please examine the outputs and identify the highest **mIoU** achieved during training. The mIoU may range between 40.45% and 60%. Kindly report the best **mIoU** obtained.
- **Step 3:**  Please identify the epoch number corresponding to the best **mIoU**. You will need to review the log to determine the epoch that yields the highest value.

You may proceed to **Part#2** of this assignment upon completion of the current task.